# Convergence Simulation — UCB vs Random Search from Cold Start

Monte Carlo simulation comparing UCB acquisition against random baseline from a cold start (zero historical runs).
Uses an accuracy-oriented Random Forest oracle (trained on all 3,400 yelp2018 MF runs) as simulated ground truth.

**Question:** From a cold start on a 20K-cell categorical space with ~5 dimensions, how many UCB-guided runs to reach ESM 99% vs. random search?

In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

from src.visualization import set_themes
from src.utils import get_kernel_id
from src.utils.config import parse_score_metric, Log10Transformer, Log2Transformer, model_based_parse_parameters, parse_parameters
from src.utils.styled_tables import data_preview_styled
from IPython.display import Markdown

from typing import List, Dict, Tuple, Optional
import io
import contextlib
from tqdm import tqdm

set_themes()
pl.Config.set_tbl_rows(20)

print("Current kernel:", get_kernel_id())

Current kernel: f0de51e5-1d5d-48ce-bdbf-5da16e07aca2


## Configuration

Define the target metric and the discrete hyperparameter space to enumerate over.

In [ ]:
run_summary_file = "wandb/summary.parquet"

target_spec = "epoch/test_recall@20"
parsed_target = parse_score_metric(target_spec)
target_metric = "target"

feature_names = [
    "embedding_dimension",
    "shuffle",
    "l1_regularization",
    "l2_regularization",
    "embedding_dropout_rate",
]

## Data Loading

Load completed runs from the local parquet cache produced by `wandb/sync.py`.

In [3]:
experiment_runs = pl.read_parquet(run_summary_file)
experiment_runs = experiment_runs.filter(pl.col("model") == "matrix_factorization")

score_expression = sum(pl.col(m) * w for m, w in parsed_target.items())
experiment_runs = experiment_runs.with_columns(
    score_expression.list.max().alias(target_metric)
)

parameter_space = {}
for col in feature_names:
    unique_vals = sorted(experiment_runs[col].drop_nulls().unique().to_list())
    if col == "shuffle":
        unique_vals = [bool(v) for v in unique_vals]
    parameter_space[col] = unique_vals

total_space_size = 1
for values in parameter_space.values():
    total_space_size *= len(values)

print(f"Loaded {len(experiment_runs):,} runs")
print(f"Total joint parameter space: {total_space_size:,} configurations")
for col, vals in parameter_space.items():
    print(f"  {col}: {vals}")

Loaded 3,410 runs
Total joint parameter space: 20,000 configurations
  embedding_dimension: [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
  shuffle: [False, True]
  l1_regularization: [0.0, 1e-10, 1e-09, 1e-08, 1e-07, 1e-06, 1e-05, 0.0001, 0.001, 0.01]
  l2_regularization: [0.0, 1e-10, 1e-09, 1e-08, 1e-07, 1e-06, 1e-05, 0.0001, 0.001, 0.01]
  embedding_dropout_rate: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


## Data Preparation

In [4]:
required_cols = feature_names + [target_metric]
full_dataframe = experiment_runs.select(required_cols).drop_nulls()

full_dataframe = full_dataframe.with_columns(pl.col("shuffle").cast(pl.Boolean))

full_features = full_dataframe.select(feature_names)
full_target = full_dataframe[target_metric].to_numpy()

print(f"Training samples: {len(full_target):,}")
print(f"Target range:  [{full_target.min():.4f}, {full_target.max():.4f}]")
print(f"Target mean:   {full_target.mean():.4f}  ±  {full_target.std():.4f}")

Training samples: 3,410
Target range:  [0.0004, 0.0511]
Target mean:   0.0217  ±  0.0193


## Full Parameter Grid

Build the complete 20,000-cell grid and tag cells as explored or unexplored based on the 3,410 completed runs.

In [5]:
all_combinations = list(itertools.product(*parameter_space.values()))
full_grid = pl.DataFrame(
    {col: [row[i] for row in all_combinations] for i, col in enumerate(feature_names)}
)
full_grid = full_grid.with_columns(pl.col("shuffle").cast(pl.Boolean))

explored_dataframe = (
    full_dataframe
    .group_by(feature_names)
    .agg(pl.col(target_metric).mean().alias(f"observed_{target_metric}"))
)
explored_keys = set(
    tuple(row) for row in explored_dataframe.select(feature_names).to_numpy().tolist()
)
full_grid = full_grid.with_columns(
    pl.struct(feature_names)
    .map_elements(lambda s: tuple(s[c] for c in feature_names) in explored_keys, return_dtype=pl.Boolean)
    .alias("explored")
)
full_grid = full_grid.join(explored_dataframe, on=feature_names, how="left")

n_explored = full_grid["explored"].sum()
n_unexplored = len(full_grid) - n_explored
coverage_pct = 100 * n_explored / len(full_grid)

print(f"Total cells:  {len(full_grid):,}")
print(f"Explored:     {n_explored:,}  ({coverage_pct:.2f}%)")
print(f"Unexplored:   {n_unexplored:,}")

Total cells:  20,000
Explored:     1,516  (7.58%)
Unexplored:   18,484


## Oracle Model Training

Retrain the oracle with finalized hyperparams from notebook 2. This provides simulated ground truth — the oracle predicts the "true" test_recall@20 for any config.

**Final hyperparams:** `max_features=None`, `max_samples=None`, `min_samples_leaf=3`, `n_estimators=1024`

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("shuffle", FunctionTransformer(lambda x: x, validate=True, feature_names_out="one-to-one"), ["shuffle"]),
        ("log2", Log2Transformer(inline_replace=False), ["embedding_dimension"]),
        ("log10", Log10Transformer(inline_replace=True), ["l1_regularization", "l2_regularization"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

oracle = Pipeline([
    ("preprocess", preprocessor),
    ("random_forest", RandomForestRegressor(
        n_estimators=1024,
        max_features=None,
        max_samples=None,
        min_samples_leaf=3,
        oob_score=True,
        n_jobs=-1,
        random_state=42,
    )),
])

sklearn.set_config(transform_output="pandas")
oracle.fit(full_features, full_target)
print(f"OOB R²: {oracle.named_steps['random_forest'].oob_score_:.4f}")


OOB R²: 0.9989


## Global Best Score

Compute the oracle's prediction across the full 20,000-cell grid to determine the true global optimum. This is used as the denominator for regret calculations in the simulation.

In [7]:
grid_features = full_grid.select(feature_names)
grid_predictions = oracle.predict(grid_features.to_pandas())

best_idx = grid_predictions.argmax()
best_score = grid_predictions[best_idx]
best_config = full_grid.row(best_idx)
best_config_tuple = tuple(best_config[full_grid.columns.index(c)] for c in feature_names)
already_explored = best_config_tuple in explored_keys

print(f"Global optimum oracle score: {best_score:.6f}")
print(f"Best config (index {best_idx}):")
for col in feature_names:
    print(f"  {col}: {best_config[full_grid.columns.index(col)]}")
print(f"  Explored: {'YES' if already_explored else 'NO'}")
print()
print(f"Total grid cells scored: {len(grid_predictions):,}")

Global optimum oracle score: 0.050729
Best config (index 16440):
  embedding_dimension: 512
  shuffle: False
  l1_regularization: 1e-07
  l2_regularization: 1e-07
  embedding_dropout_rate: 0.0
  Explored: YES

Total grid cells scored: 20,000


## Shared Experiment Functions

Define reusable runner and plotting functions. Each experiment below only needs to define `CONFIGS` and call these functions.

In [8]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

from src.utils.simulation_worker import run_trajectory

MAX_WORKERS = 16
output_directory = "wandb/trajectories"
probe_interval = 10

# Construct parameters_config for model_based_parse_parameters
parameters_config = {
    "model": {"value": "matrix_factorization"},
}
for col in feature_names:
    parameters_config[col] = {
        "distribution": "categorical",
        "values": parameter_space[col],
    }


def run_experiment(configs: List[Dict], n_runs: int = 1000, n_trajectories: int = 3):
    all_futures = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS, mp_context=get_context("spawn")) as executor:
        for config in configs:
            for seed in range(n_trajectories):
                all_futures.append(executor.submit(
                    run_trajectory,
                    strategy=config["strategy"],
                    seed=seed,
                    parameters_config=parameters_config,
                    oracle_pipeline=oracle,
                    feature_names=feature_names,
                    target_spec=target_spec,
                    global_best=best_score,
                    n_runs=n_runs,
                    beta=config.get("beta", 1.0),
                    estimator_count=config.get("estimator_count", 128),
                    virtual_sample_count=config.get("virtual_sample_count", 100),
                    virtual_lambda=config.get("virtual_lambda", 2.0),
                    max_samples=config.get("max_samples", 0.1),
                    min_samples_leaf=config.get("min_samples_leaf", 3),
                    probe_interval=probe_interval,
                    output_directory=output_directory,
                    label=config.get("label"),
                    verbose=True,
                ))

        all_history = [future.result() for future in as_completed(all_futures)]

    return pl.concat(all_history)


def filter_config(data: pl.DataFrame, config: Dict, group_keys: Tuple[str, ...]) -> pl.DataFrame:
    for key in group_keys:
        data = data.filter(pl.col(key) == config[key])
    return data


def plot_experiment(data: pl.DataFrame, configs: List[Dict], global_optimum: float, group_keys: Tuple[str, ...] = ("strategy", "max_samples")):
    from matplotlib.figure import Figure
    from matplotlib.axes import Axes

    fig: Figure
    axes: list[Axes]
    fig, axes = plt.subplots(4, 1, figsize=(10, 18))

    seeds = sorted(data["seed"].unique().to_list())
    n_runs_plot = data["run"].max() + 1

    aggregated = (
        data
        .group_by(["run"] + list(group_keys))
        .agg([
            pl.col("best_found").mean().alias("bf_mean"),
            pl.col("best_found").std(ddof=0).alias("bf_std"),
            pl.col("simple_regret").mean().alias("sr_mean"),
            pl.col("simple_regret").std(ddof=0).alias("sr_std"),
            pl.col("score").mean().alias("score_mean"),
            pl.col("score").std(ddof=0).alias("score_std"),
            pl.col("esm").drop_nulls().mean().alias("esm_mean"),
            pl.col("esm").drop_nulls().std(ddof=0).alias("esm_std"),
        ])
        .sort(["run"] + list(group_keys))
    )

    # Plot 1: Best found over runs
    ax: Axes = axes[0]
    for config in configs:
        agg = filter_config(aggregated, config, group_keys).sort("run")
        seed_trajs = filter_config(data, config, group_keys)
        for seed in seeds:
            traj = seed_trajs.filter(pl.col("seed") == seed).sort("run")
            ax.plot(traj["run"], traj["best_found"], color=config["color"], linewidth=0.5, alpha=0.2)
        ax.plot(agg["run"], agg["bf_mean"], label=config["label"], color=config["color"], linewidth=1.5)
        ax.fill_between(agg["run"], agg["bf_mean"] - agg["bf_std"], agg["bf_mean"] + agg["bf_std"],
                         color=config["color"], alpha=0.12)
    ax.axhline(global_optimum, color="green", linestyle="--", linewidth=1, alpha=0.7, label=f"Optimum ({global_optimum:.6f})")
    ax.set_xlabel("Run")
    ax.set_ylabel("Best found score")
    ax.set_title("Convergence (mean ± 1σ, per-seed in faded)")
    ax.legend(fontsize=9)
    ax.set_xlim(0, n_runs_plot - 1)

    # Plot 2: Simple regret over runs
    ax = axes[1]
    for config in configs:
        agg = filter_config(aggregated, config, group_keys).sort("run")
        seed_trajs = filter_config(data, config, group_keys)
        for seed in seeds:
            traj = seed_trajs.filter(pl.col("seed") == seed).sort("run")
            ax.plot(traj["run"], traj["simple_regret"], color=config["color"], linewidth=0.5, alpha=0.2)
        ax.plot(agg["run"], agg["sr_mean"], label=config["label"], color=config["color"], linewidth=1.5)
        ax.fill_between(agg["run"], agg["sr_mean"] - agg["sr_std"], agg["sr_mean"] + agg["sr_std"],
                         color=config["color"], alpha=0.12)
    ax.axhline(0, color="green", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xlabel("Run")
    ax.set_ylabel("Simple regret")
    ax.set_title("Simple Regret (mean ± 1σ)")
    ax.legend(fontsize=9)
    ax.set_xlim(0, n_runs_plot - 1)

    # Plot 3: Per-run score with rolling mean
    ax = axes[2]
    score_window = 50
    for config in configs:
        agg = filter_config(aggregated, config, group_keys).sort("run")
        seed_trajs = filter_config(data, config, group_keys)
        for seed in seeds:
            traj = seed_trajs.filter(pl.col("seed") == seed).sort("run")
            rolling_seed = traj.select(
                pl.col("score").rolling_mean(window_size=score_window, min_samples=1)
            ).to_series().to_numpy()
            ax.plot(traj["run"], rolling_seed, color=config["color"], linewidth=0.5, alpha=0.2)
        rolling_mean = agg.select(
            pl.col("score_mean").rolling_mean(window_size=score_window, min_samples=1)
        ).to_series().to_numpy()
        rolling_std = agg.select(
            pl.col("score_std").rolling_mean(window_size=score_window, min_samples=1)
        ).to_series().to_numpy()
        ax.plot(agg["run"], rolling_mean, label=f"{config['label']} (MA{score_window})", color=config["color"], linewidth=1.5)
        ax.fill_between(agg["run"], rolling_mean - rolling_std, rolling_mean + rolling_std,
                         color=config["color"], alpha=0.12)
    ax.axhline(global_optimum, color="green", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xlabel("Run")
    ax.set_ylabel("Score (MA20)")
    ax.set_title(f"Per-Run Score (MA{score_window}, mean ± 1σ)")
    ax.legend(fontsize=9)
    ax.set_xlim(0, n_runs_plot - 1)

    # Plot 4: ESM over runs
    ax = axes[3]
    esm_window = 20
    for config in configs:
        seed_trajs = filter_config(data, config, group_keys)
        for seed in seeds:
            traj = seed_trajs.filter(pl.col("seed") == seed).sort("run")
            esm_data = traj.filter(pl.col("esm").is_not_null())
            if len(esm_data) > 0:
                rolling_esm_seed = esm_data.select(
                    pl.col("esm").rolling_mean(window_size=max(1, len(esm_data) // 10), min_samples=1)
                ).to_series().to_numpy()
                ax.plot(esm_data["run"], rolling_esm_seed, color=config["color"], linewidth=0.5, alpha=0.2)
        agg_esm = filter_config(aggregated, config, group_keys).filter(pl.col("esm_mean").is_not_null()).sort("run")
        if len(agg_esm) > 0:
            rolling_esm = agg_esm.select(
                pl.col("esm_mean").rolling_mean(window_size=esm_window, min_samples=1)
            ).to_series().to_numpy()
            rolling_esm_std = agg_esm.select(
                pl.col("esm_std").rolling_mean(window_size=esm_window, min_samples=1)
            ).to_series().to_numpy()
            ax.plot(agg_esm["run"], rolling_esm, label=config["label"], color=config["color"], linewidth=1.5)
            ax.fill_between(agg_esm["run"], rolling_esm - rolling_esm_std, rolling_esm + rolling_esm_std,
                             color=config["color"], alpha=0.12)
    ax.axhline(100, color="green", linestyle="--", linewidth=1, alpha=0.7, label="100% saturation")
    ax.set_xlabel("Run")
    ax.set_ylabel("ESM (%)")
    ax.set_title("Exploration Saturation (MA20, mean ± 1σ)")
    ax.legend(fontsize=9)
    ax.set_xlim(0, n_runs_plot - 1)

    plt.tight_layout()
    plt.show()

## Experiment 1 — min_samples_leaf sweep

Comparing `min_samples_leaf ∈ {1, 2, 3, 5, 10}` for UCB acquisition. Random baseline at leaf=3. All with `max_samples=0.1`, `virtual_lambda=1.0`, β=1.0.

In [ ]:
CONFIGS = [
    {"strategy": "random", "min_samples_leaf": 3,  "max_samples": 0.1, "virtual_lambda": 1.0, "label": "Random",               "color": "#ca6702"},
    {"strategy": "ucb",    "min_samples_leaf": 1,  "max_samples": 0.1, "virtual_lambda": 1.0, "label": "UCB (leaf=1)",         "color": "#d62828"},
    {"strategy": "ucb",    "min_samples_leaf": 2,  "max_samples": 0.1, "virtual_lambda": 1.0, "label": "UCB (leaf=2)",         "color": "#f77f00"},
    {"strategy": "ucb",    "min_samples_leaf": 3,  "max_samples": 0.1, "virtual_lambda": 1.0, "label": "UCB (leaf=3)",         "color": "#2e86ab"},
    {"strategy": "ucb",    "min_samples_leaf": 5,  "max_samples": 0.1, "virtual_lambda": 1.0, "label": "UCB (leaf=5)",         "color": "#2d6a4f"},
    {"strategy": "ucb",    "min_samples_leaf": 10, "max_samples": 0.1, "virtual_lambda": 1.0, "label": "UCB (leaf=10)",        "color": "#7209b7"},
]
N_TRAJECTORIES = 3
N_RUNS = 1000

n_total_runs = len(CONFIGS) * N_TRAJECTORIES * N_RUNS
print(f"Configs: {len(CONFIGS)} \u00d7 {N_TRAJECTORIES} seeds \u00d7 {N_RUNS} runs = {n_total_runs:,} total")

leaf_experiment_results: pl.DataFrame = run_experiment(CONFIGS, n_runs=N_RUNS, n_trajectories=N_TRAJECTORIES)

In [ ]:
n_total = len(leaf_experiment_results)
print(f"Total records : {n_total:,}")
print(f"Seeds        : {sorted(leaf_experiment_results['seed'].unique().to_list())}")
print(f"Leaf values  : {sorted(leaf_experiment_results['min_samples_leaf'].unique().to_list())}")
print()

summary = (
    leaf_experiment_results
    .group_by(["strategy", "min_samples_leaf", "seed"])
    .agg([
        pl.col("run").max().alias("runs"),
        pl.col("simple_regret").last().alias("final_regret"),
        pl.col("best_found").last().alias("best"),
        pl.col("esm").drop_nulls().last().alias("esm"),
    ])
    .sort(["strategy", "min_samples_leaf", "seed"])
)
print(summary)

plot_experiment(leaf_experiment_results, CONFIGS, best_score, group_keys=("strategy", "min_samples_leaf"))

## Experiment 2 — beta sweep

Comparing `β ∈ {0.5, 1.0, 2.0, 4.0}` for UCB acquisition. Random baseline at β=1.0. All with `max_samples=0.1`, `virtual_lambda=1.0`, `min_samples_leaf=3`.

In [ ]:
CONFIGS = [
    {"strategy": "random", "beta": 1.0, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "Random",            "color": "#ca6702"},
    {"strategy": "ucb",    "beta": 0.5, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (\u03b2=0.5)",       "color": "#d62828"},
    {"strategy": "ucb",    "beta": 1.0, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (\u03b2=1.0)",       "color": "#f77f00"},
    {"strategy": "ucb",    "beta": 2.0, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (\u03b2=2.0)",       "color": "#2e86ab"},
    {"strategy": "ucb",    "beta": 4.0, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (\u03b2=4.0)",       "color": "#2d6a4f"},
]
N_TRAJECTORIES = 3
N_RUNS = 1000

n_total_runs = len(CONFIGS) * N_TRAJECTORIES * N_RUNS
print(f"Configs: {len(CONFIGS)} \u00d7 {N_TRAJECTORIES} seeds \u00d7 {N_RUNS} runs = {n_total_runs:,} total")

beta_experiment_results: pl.DataFrame = run_experiment(CONFIGS, n_runs=N_RUNS, n_trajectories=N_TRAJECTORIES)

In [ ]:
n_total = len(beta_experiment_results)
print(f"Total records : {n_total:,}")
print(f"Seeds        : {sorted(beta_experiment_results['seed'].unique().to_list())}")
print(f"Beta values  : {sorted(beta_experiment_results['beta'].unique().to_list())}")
print()

summary = (
    beta_experiment_results
    .group_by(["strategy", "beta", "seed"])
    .agg([
        pl.col("run").max().alias("runs"),
        pl.col("simple_regret").last().alias("final_regret"),
        pl.col("best_found").last().alias("best"),
        pl.col("esm").drop_nulls().last().alias("esm"),
    ])
    .sort(["strategy", "beta", "seed"])
)
print(summary)

plot_experiment(beta_experiment_results, CONFIGS, best_score, group_keys=("strategy", "beta"))

## Experiment 3 — lambda sweep

Comparing `virtual_lambda ∈ {0.5, 1.0, 2.0}` for UCB acquisition. Random baseline at λ=1.0. All with `max_samples=0.1`, `min_samples_leaf=3`, β=1.0.

In [ ]:
CONFIGS = [
    {"strategy": "random", "virtual_lambda": 1.0, "max_samples": 0.1, "min_samples_leaf": 3, "label": "Random",          "color": "#ca6702"},
    {"strategy": "ucb",    "virtual_lambda": 0.5, "max_samples": 0.1, "min_samples_leaf": 3, "label": "UCB (\u03bb=0.5)",     "color": "#d62828"},
    {"strategy": "ucb",    "virtual_lambda": 1.0, "max_samples": 0.1, "min_samples_leaf": 3, "label": "UCB (\u03bb=1.0)",     "color": "#f77f00"},
    {"strategy": "ucb",    "virtual_lambda": 2.0, "max_samples": 0.1, "min_samples_leaf": 3, "label": "UCB (\u03bb=2.0)",     "color": "#2e86ab"},
]
N_TRAJECTORIES = 3
N_RUNS = 10000

n_total_runs = len(CONFIGS) * N_TRAJECTORIES * N_RUNS
print(f"Configs: {len(CONFIGS)} \u00d7 {N_TRAJECTORIES} seeds \u00d7 {N_RUNS} runs = {n_total_runs:,} total")

lambda_experiment_results: pl.DataFrame = run_experiment(CONFIGS, n_runs=N_RUNS, n_trajectories=N_TRAJECTORIES)

In [ ]:
n_total = len(lambda_experiment_results)
print(f"Total records : {n_total:,}")
print(f"Seeds        : {sorted(lambda_experiment_results['seed'].unique().to_list())}")
print(f"Lambda values: {sorted(lambda_experiment_results['virtual_lambda'].unique().to_list())}")
print()

summary = (
    lambda_experiment_results
    .group_by(["strategy", "virtual_lambda", "seed"])
    .agg([
        pl.col("run").max().alias("runs"),
        pl.col("simple_regret").last().alias("final_regret"),
        pl.col("best_found").last().alias("best"),
        pl.col("esm").drop_nulls().last().alias("esm"),
    ])
    .sort(["strategy", "virtual_lambda", "seed"])
)
print(summary)

plot_experiment(lambda_experiment_results, CONFIGS, best_score, group_keys=("strategy", "virtual_lambda"))

### ESM Convergence Analysis (MA20)

Smooths ESM with a 20-run moving average per trajectory, then measures how much time is spent above 99% — a more robust view than raw per-run thresholds.

| λ | First run ESM ≥ 99% | Runs at ≥ 99% | % of total |
|---|-------------------|--------------|-----------|
| 0.5 | ~1,888 | 839 | 8.4% |
| 1.0 | ~5,314 | 515 | 5.1% |
| 2.0 | ~8,752 | 10 | 0.1% |

**Insight:** Even λ=0.5 (the best performer) only stays above 99% for ~8% of the trajectory. Most runs sit at ~98% ESM. This is because **ESM measures grid coverage, not search convergence** — `best_ucb ≈ best_observed / 0.98` means the surrogate honestly believes the best unexplored cell among ~19,360 remaining cells could be ~2% better than what's been found. That's a reasonable assessment given only ~3.2% of the 20K grid has been explored by UCB λ=0.5.

In [ ]:
lambda_experiment_results.filter(
    pl.col("strategy") == "ucb",
).with_columns(
    pl.col("esm")
    .rolling_mean(window_size=20, min_samples=1)
    .over(["strategy", "virtual_lambda", "seed"])
    .alias("esm_ma20")
).group_by(["strategy", "virtual_lambda", "seed"]).agg(
    esm99_count = pl.col("esm_ma20").filter(pl.col("esm_ma20") >= 99).count(),
    esm99_ratio = pl.col("esm_ma20").filter(pl.col("esm_ma20") >= 99).count() / pl.col("esm_ma20").count(),
    first_esm_99 = pl.col("run").filter(pl.col("esm_ma20") >= 99).min().alias("first_esm_99"),
).group_by(["strategy", "virtual_lambda"]).agg(
    esm99_count = pl.col("esm99_count").mean(),
    esm99_ratio = pl.col("esm99_ratio").mean(),
    first_esm_99 = pl.col("first_esm_99").mean(),
).sort("virtual_lambda")

## Experiment 4 — max_samples sweep

Comparing `max_samples ∈ {0.1, 0.5, 1.0}` for UCB acquisition. Random baseline at ms=0.1. All with `virtual_lambda=1.0`, `min_samples_leaf=3`, β=1.0.

In [ ]:
CONFIGS = [
    {"strategy": "random", "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "Random",              "color": "#ca6702"},
    {"strategy": "ucb",    "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (ms=0.1)",        "color": "#d62828"},
    {"strategy": "ucb",    "max_samples": 0.5, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (ms=0.5)",        "color": "#f77f00"},
    {"strategy": "ucb",    "max_samples": 1.0, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (ms=1.0)",        "color": "#2e86ab"},
]
N_TRAJECTORIES = 3
N_RUNS = 1000

n_total_runs = len(CONFIGS) * N_TRAJECTORIES * N_RUNS
print(f"Configs: {len(CONFIGS)} \u00d7 {N_TRAJECTORIES} seeds \u00d7 {N_RUNS} runs = {n_total_runs:,} total")

max_samples_experiment_results: pl.DataFrame = run_experiment(CONFIGS, n_runs=N_RUNS, n_trajectories=N_TRAJECTORIES)

In [ ]:
n_total = len(max_samples_experiment_results)
print(f"Total records : {n_total:,}")
print(f"Seeds        : {sorted(max_samples_experiment_results['seed'].unique().to_list())}")
print(f"Max samples  : {sorted(max_samples_experiment_results['max_samples'].unique().to_list())}")
print()

summary = (
    max_samples_experiment_results
    .group_by(["strategy", "max_samples", "seed"])
    .agg([
        pl.col("run").max().alias("runs"),
        pl.col("simple_regret").last().alias("final_regret"),
        pl.col("best_found").last().alias("best"),
        pl.col("esm").drop_nulls().last().alias("esm"),
    ])
    .sort(["strategy", "max_samples", "seed"])
)
print(summary)

plot_experiment(max_samples_experiment_results, CONFIGS, best_score)

## Experiment 5 — n_estimators sweep

Comparing `n_estimators ∈ {128, 256, 512, 1024}` for UCB acquisition. All with `max_samples=0.1`, `virtual_lambda=1.0`, `min_samples_leaf=3`, β=1.0.

In [ ]:
CONFIGS = [
    {"strategy": "random", "estimator_count": 128, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "Random",                  "color": "#ca6702"},
    {"strategy": "ucb",    "estimator_count": 128, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (128 trees)",        "color": "#d62828"},
    {"strategy": "ucb",    "estimator_count": 256, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (256 trees)",        "color": "#f77f00"},
    {"strategy": "ucb",    "estimator_count": 512, "max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (512 trees)",        "color": "#2e86ab"},
    {"strategy": "ucb",    "estimator_count": 1024,"max_samples": 0.1, "virtual_lambda": 1.0, "min_samples_leaf": 3, "label": "UCB (1024 trees)",       "color": "#2d6a4f"},
]
N_TRAJECTORIES = 3
N_RUNS = 1000

n_total_runs = len(CONFIGS) * N_TRAJECTORIES * N_RUNS
print(f"Configs: {len(CONFIGS)} \u00d7 {N_TRAJECTORIES} seeds \u00d7 {N_RUNS} runs = {n_total_runs:,} total")

estimator_experiment_results: pl.DataFrame = run_experiment(CONFIGS, n_runs=N_RUNS, n_trajectories=N_TRAJECTORIES)

In [ ]:
n_total = len(estimator_experiment_results)
print(f"Total records : {n_total:,}")
print(f"Seeds        : {sorted(estimator_experiment_results['seed'].unique().to_list())}")
print(f"Trees        : {sorted(estimator_experiment_results['estimator_count'].unique().to_list())}")
print()

summary = (
    estimator_experiment_results
    .group_by(["strategy", "estimator_count", "seed"])
    .agg([
        pl.col("run").max().alias("runs"),
        pl.col("simple_regret").last().alias("final_regret"),
        pl.col("best_found").last().alias("best"),
        pl.col("esm").drop_nulls().last().alias("esm"),
    ])
    .sort(["strategy", "estimator_count", "seed"])
)
print(summary)

plot_experiment(estimator_experiment_results, CONFIGS, best_score, group_keys=("strategy", "estimator_count"))